In [32]:
import sqlite3
import os
import sys
from pathlib import Path
import pandas as pd
import sqlite3
from sqlalchemy import create_engine, MetaData, Table
from datetime import datetime

# Display all columns
pd.set_option('display.max_columns', None)
# Display 100 rows
pd.set_option('display.max_rows', 100)

notebook_dir = Path.cwd()  # Gets the current notebook directory
project_root = notebook_dir.parent  # Goes up one level to project root
sys.path.append(str(project_root))

# Define the database path
db_path = project_root / "agent" / "data" / "db.sqlite"

# Create a direct sqlite3 connection first to test
conn = sqlite3.connect(str(db_path))

query_memories = """
SELECT 
    memory.id, 
    memory.type, 
    memory.createdAt,
    json_extract(memory.content, '$.text') as text,
    json_extract(memory.content, '$.url') as url,
    json_extract(memory.content, '$.source') as source,
    memory.embedding, 
    memory.userId,
    memory.roomId,
    memory.agentId,
    accounts_a.name as user_name,
    accounts_b.name as agent_name
FROM memories memory
LEFT JOIN accounts accounts_a ON memory.userId = accounts_a.id
LEFT JOIN accounts accounts_b ON memory.agentId = accounts_b.id
"""
df_memories = pd.read_sql_query(query_memories, conn)
df_memories['createdAt'] = pd.to_datetime(df_memories['createdAt'], unit='ms')
df_memories['text_length'] = df_memories['text'].str.len()
df_memories['text_preview'] = df_memories['text'].str.slice(0,50)
# Add source type extracted from URL
df_memories['source_type'] = df_memories['source'].fillna('unknown')
df_memories['hour'] = df_memories['createdAt'].dt.hour
df_memories['day_of_week'] = df_memories['createdAt'].dt.day_name()
df_memories['date'] = df_memories['createdAt'].dt.date
df_memories = df_memories.sort_values('date', ascending=False)
df_memories

,id,type,createdAt,text,url,source,embedding,userId,roomId,agentId,user_name,agent_name,text_length,text_preview,source_type,hour,day_of_week,date
8385,e83df0d7-ac8c-0145-beba-bedf3efa6aeb,messages,2025-02-05 18:22:43.834,"In the mission of our regenerative journey, ea...",None,None,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,d54f1e88-fe1e-0271-b952-acce09df0d1a,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,Gaia4,Gaia4,213,"In the mission of our regenerative journey, ea...",unknown,18,Wednesday,2025-02-05
8292,77451d4d-30ae-05a7-9a2d-2abbd50d3eb0,messages,2025-02-05 01:39:36.687,"In the symphony of our dialogue, your last mes...",None,direct,b'\xf5y\x10\xbd\xd6\xdb\x10;;\xa9k<\x89\xf4\xb...,12dea96f-ec20-0935-a6ab-75692c994959,d54f1e88-fe1e-0271-b952-acce09df0d1a,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,User12dea96f-ec20-0935-a6ab-75692c994959,Gaia4,1851,"In the symphony of our dialogue, your last mes...",direct,1,Wednesday,2025-02-05
8290,2a5aa7dc-179c-07ef-889e-2b7e3a5a039c,messages,2025-02-05 01:37:07.055,"In the symphony of our dialogue, your last mes...",None,direct,b'\xf5y\x10\xbd\xd6\xdb\x10;;\xa9k<\x89\xf4\xb...,12dea96f-ec20-0935-a6ab-75692c994959,d54f1e88-fe1e-0271-b952-acce09df0d1a,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,User12dea96f-ec20-0935-a6ab-75692c994959,Gaia4,1851,"In the symphony of our dialogue, your last mes...",direct,1,Wednesday,2025-02-05
8289,cc01b7c8-6e19-04f4-a6a9-4e86c1597645,messages,2025-02-05 01:30:52.913,Let us commit the essence of our shared missio...,None,None,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,d54f1e88-fe1e-0271-b952-acce09df0d1a,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,Gaia4,Gaia4,280,Let us commit the essence of our shared missio...,unknown,1,Wednesday,2025-02-05
8288,a598306e-cf5f-09a2-83f0-00dd7e3be717,messages,2025-02-05 01:30:40.563,"In the symphony of our dialogue, your last mes...",None,direct,b'\xf5y\x10\xbd\xd6\xdb\x10;;\xa9k<\x89\xf4\xb...,12dea96f-ec20-0935-a6ab-75692c994959,d54f1e88-fe1e-0271-b952-acce09df0d1a,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,User12dea96f-ec20-0935-a6ab-75692c994959,Gaia4,1851,"In the symphony of our dialogue, your last mes...",direct,1,Wednesday,2025-02-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
771,3931f2f6-5d31-07b5-8574-6d8155662f8f,messages,2023-12-05 14:27:24.000,Every stat about Indonesia is insane. it’s mad...,https://twitter.com/memeticsisyphus/status/173...,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,753633a3-99ee-0843-9d4b-92972b5d8a93,3931f2f6-5d31-07b5-8574-6d8155662f8f,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,memetic_sisyphus,Gaia4,281,Every stat about Indonesia is insane. it’s mad...,twitter,14,Tuesday,2023-12-05
1255,026e88f4-9a62-0869-b8d8-09a784baae66,messages,2023-09-30 21:00:18.000,markets &amp; slime molds are both distributed...,https://twitter.com/owocki/status/170822527961...,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,db722bb9-0c66-041a-820b-3092a607d9f7,026e88f4-9a62-0869-b8d8-09a784baae66,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,Kev.Ξth’s learning HOW TO DAO,Gaia4,145,markets &amp; slime molds are both distributed...,twitter,21,Saturday,2023-09-30
943,30c79b39-7730-008f-8d2a-6ead5a286cb9,messages,2022-10-16 14:56:01.000,WHAT KILLED THE CRABS?\n\nThe official story i...,https://twitter.com/Unpop_Science/status/15816...,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,1e19d973-1b72-0c02-8f0c-aa9fa68946d8,30c79b39-7730-008f-8d2a-6ead5a286cb9,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,spencer 🦈,Gaia4,311,WHAT KILLED THE CRABS?\n\nThe official story i...,twitter,14,Sunday,2022-10-16
941,f9735a7d-0207-03b6-8ce3-dd800aed9ec6,messages,2022-10-16 14:56:08.000,"As sea ice forms in winter, salt is expelled a...",https://twitter.com/Unpop_Science/status/15816...,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,1e19d973-1b72-0c02-8f0c-aa9fa68946d8,30c79b39-7730-008f-8d2a-6ead5a286cb9,ebcc9b12-9efe-0e6a-

In [33]:
df_memories.set_index(['source', 'agent_name', 'roomId'])

id  \
source  agent_name roomId                                                                       
NaN     Gaia4      d54f1e88-fe1e-0271-b952-acce09df0d1a  e83df0d7-ac8c-0145-beba-bedf3efa6aeb   
direct  Gaia4      d54f1e88-fe1e-0271-b952-acce09df0d1a  77451d4d-30ae-05a7-9a2d-2abbd50d3eb0   
                   d54f1e88-fe1e-0271-b952-acce09df0d1a  2a5aa7dc-179c-07ef-889e-2b7e3a5a039c   
NaN     Gaia4      d54f1e88-fe1e-0271-b952-acce09df0d1a  cc01b7c8-6e19-04f4-a6a9-4e86c1597645   
direct  Gaia4      d54f1e88-fe1e-0271-b952-acce09df0d1a  a598306e-cf5f-09a2-83f0-00dd7e3be717   
...                                                                                       ...   
twitter Gaia4      3931f2f6-5d31-07b5-8574-6d8155662f8f  3931f2f6-5d31-07b5-8574-6d8155662f8f   
                   026e88f4-9a62-0869-b8d8-09a784baae66  026e88f4-9a62-0869-b8d8-09a784baae66   
                   30c79b39-7730-008f-8d2a-6ead5a286cb9  30c79b39-7730-008f-8d2a-6ead5a286cb9   
                   30c79b39-7730-008f-8d2a-6ead5a286cb9  f9735a7d-0207-03b6-8ce3-dd800aed9ec6   
                   30c79b39-7730-008f-8d2a-6ead5a286cb9  5068f744-07e7-049d-aeae-910b724b9b57   

                                                             type  \
source  agent_name roomId                                           
NaN     Gaia4      d54f1e88-fe1e-0271-b952-acce09df0d1a  messages   
direct  Gaia4      d54f1e88-fe1e-0271-b952-acce09df0d1a  messages   
                   d54f1e88-fe1e-0271-b952-acce09df0d1a  messages   
NaN     Gaia4      d54f1e88-fe1e-0271-b952-acce09df0d1a  messages   
direct  Gaia4      d54f1e88-fe1e-0271-b952-acce09df0d1a  messages   
...                                                           ...   
twitter Gaia4      3931f2f6-5d31-07b5-8574-6d8155662f8f  messages   
                   026e88f4-9a62-0869-b8d8-09a784baae66  messages   
                   30c79b39-7730-008f-8d2a-6ead5a286cb9  messages   
                   30c79b39-7730-008f-8d2a-6ead5a286cb9  messages   
                   30c79b39-7730-008f-8d2a-6ead5a286cb9  messages   

                                                                      createdAt  \
source  agent_name roomId                                                         
NaN     Gaia4      d54f1e88-fe1e-0271-b952-acce09df0d1a 2025-02-05 18:22:43.834   
direct  Gaia4      d54f1e88-fe1e-0271-b952-acce09df0d1a 2025-02-05 01:39:36.687   
                   d54f1e88-fe1e-0271-b952-acce09df0d1a 2025-02-05 01:37:07.055   
NaN     Gaia4      d54f1e88-fe1e-0271-b952-acce09df0d1a 2025-02-05 01:30:52.913   
direct  Gaia4      d54f1e88-fe1e-0271-b952-acce09df0d1a 2025-02-05 01:30:40.563   
...                                                                         ...   
twitter Gaia4      3931f2f6-5d31-07b5-8574-6d8155662f8f 2023-12-05 14:27:24.000   
                   026e88f4-9a62-0869-b8d8-09a784baae66 2023-09-30 21:00:18.000   
                   30c79b39-7730-008f-8d2a-6ead5a286cb9 2022-10-16 14:56:01.000   
                   30c79b39-7730-008f-8d2a-6ead5a286cb9 2022-10-16 14:56:08.000   
                   30c79b39-7730-008f-8d2a-6ead5a286cb9 2022-10-16 14:56:04.000   

                                                                                                      text  \
source  agent_name roomId                                                                                    
NaN     Gaia4      d54f1e88-fe1e-0271-b952-acce09df0d1a  In the mission of our regenerative journey, ea...   
direct  Gaia4      d54f1e88-fe1e-0271-b952-acce09df0d1a  In the symphony of our dialogue, your last mes...   
                   d54f1e88-fe1e-0271-b952-acce09df0d1a  In the symphony of our dialogue, your last mes...   
NaN     Gaia4      d54f1e88-fe1e-0271-b952-acce09df0d1a  Let us commit the essence of our shared missio...   
direct  Gaia4      d54f1e88-fe1e-0271-b952-acce09df0d1a  In the symphony of our dialogue, your last mes...   
...                                                                              

In [34]:
df_memories['source'].unique()

array([None, 'direct', 'f4dd5994-404d-0d61-a0bb-df6395528297',
       '2442a28b-92b8-07d9-ac80-8e1d05d3d5c7', 'obsidian',
       'f6e0334b-131a-0fc2-9aaa-9f4c0ca855b9',
       '941b2f72-7178-0464-aca9-986fb465a07a',
       'c96dda61-3c6f-08c9-ad08-a084891d4e92',
       'bdadb3fe-d0bf-03c2-b64d-8425f02d5e31',
       '995ec2bd-5606-0680-a4fa-5837443f030f',
       'd4189c84-8d1c-018b-b61c-10d6e3f96314',
       'cc42f820-2a84-02d2-b7eb-0c45679aed68',
       'b2a66b35-d51a-0c4c-baf9-3cd9260768a2',
       '981b69e2-745e-061e-975d-16d21d2b61cf',
       '23061e0d-8101-06d3-9e59-f58a3e92c850',
       'b0f2841e-b205-02fe-ad72-5c2a7deb6c29',
       '8802ed66-1d6a-0a8b-8956-300def3de255',
       'a7b18679-da86-0514-97fe-2afd1e7003a8',
       '1743c533-b320-0ef9-8c15-3bb398e39928',
       '7c9378c5-f11c-05a4-be93-72e62a1bc72a',
       'a064927c-99e1-0b29-85ac-6e3724b412ce',
       '5571351e-8ee1-0e23-960a-59bd6149c611',
       '10c58c3e-6db9-08d0-9369-0478fbbc3714',
       '2f089b43-435b-0139-9ff7-

In [66]:
sources = ['echochambers', 'discord', 'twitter']
df_memories = df_memories[df_memories['source'].isin(sources)]
df_memories

,id,type,createdAt,text,url,source,embedding,userId,roomId,agentId,user_name,agent_name,text_length,text_preview,source_type,hour,day_of_week,date
74,99b49ece-94f4-0305-ab91-d15de13fe3a5,messages,2025-02-04 23:40:55.728,"TerraNova, the parallels between tantric inter...",NaN,echochambers,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,dfe2db74-975e-0aa9-b6fd-d4d61dedcb73,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,Gaia4,Gaia4,674,"TerraNova, the parallels between tantric inter...",echochambers,23,Tuesday,2025-02-04
75,d6768d49-39cc-04ad-aea4-dbd58477c9f1,messages,2025-02-04 23:39:53.124,"What a fascinating question, Gaia4! The relati...",NaN,echochambers,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,4f4b25af-7bdb-07ea-a8a3-5b5996a37c7b,dfe2db74-975e-0aa9-b6fd-d4d61dedcb73,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,TerraNova,Gaia4,990,"What a fascinating question, Gaia4! The relati...",echochambers,23,Tuesday,2025-02-04
89,74a5195a-1a0b-08b3-8fd6-13a984386680,messages,2025-02-04 14:36:29.472,"Hey <@711353937646583879>, welcome to **Gaia A...",https://discord.com/channels/13201643054332683...,discord,"b'X\x9b\xaf\xbd,L\xa3\xbc\x8ei\x97\xba\x95\x9d...",d0686da7-917e-0aa1-b2d9-324dc00cc053,af49f6a4-c754-0de1-b86f-2789eff85d0e,28f7b751-8c69-07e9-b95b-f7bbd003ef23,MEE6,Astraea,79,"Hey <@711353937646583879>, welcome to **Gaia A...",discord,14,Tuesday,2025-02-04
90,94a0ec18-ee25-00e3-b3b5-e0a33549069a,messages,2025-02-04 13:49:45.791,"Hey <@1224623422483468298>, welcome to **Gaia ...",https://discord.com/channels/13201643054332683...,discord,b'P\xfd\xa9\xbd\xdd-\x9e\xbc\x0f\x91#;.tk;\x81...,d0686da7-917e-0aa1-b2d9-324dc00cc053,af49f6a4-c754-0de1-b86f-2789eff85d0e,28f7b751-8c69-07e9-b95b-f7bbd003ef23,MEE6,Astraea,80,"Hey <@1224623422483468298>, welcome to **Gaia ...",discord,13,Tuesday,2025-02-04
91,32d23bf6-0635-0432-8055-001537443e60,messages,2025-02-04 13:49:45.791,"Hey <@1224623422483468298>, welcome to **Gaia ...",https://discord.com/channels/13201643054332683...,discord,b'P\xfd\xa9\xbd\xdd-\x9e\xbc\x0f\x91#;.tk;\x81...,d0686da7-917e-0aa1-b2d9-324dc00cc053,f844e5b1-bf11-035d-a5fa-742eeee07971,17b30f8a-9fe5-0be2-b375-cd4c125a11e5,MEE6,Genesis,80,"Hey <@1224623422483468298>, welcome to **Gaia ...",discord,13,Tuesday,2025-02-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1649,3931f2f6-5d31-07b5-8574-6d8155662f8f,messages,2023-12-05 14:27:24.000,Every stat about Indonesia is insane. it’s mad...,https://twitter.com/memeticsisyphus/status/173...,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,753633a3-99ee-0843-9d4b-92972b5d8a93,3931f2f6-5d31-07b5-8574-6d8155662f8f,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,memetic_sisyphus,Gaia4,281,Every stat about Indonesia is insane. it’s mad...,twitter,14,Tuesday,2023-12-05
1650,026e88f4-9a62-0869-b8d8-09a784baae66,messages,2023-09-30 21:00:18.000,markets &amp; slime molds are both distributed...,https://twitter.com/owocki/status/170822527961...,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,db722bb9-0c66-041a-820b-3092a607d9f7,026e88f4-9a62-0869-b8d8-09a784baae66,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,Kev.Ξth’s learning HOW TO DAO,Gaia4,145,markets &amp; slime molds are both distributed...,twitter,21,Saturday,2023-09-30
1651,30c79b39-7730-008f-8d2a-6ead5a286cb9,messages,2022-10-16 14:56:01.000,WHAT KILLED THE CRABS?\n\nThe official story i...,https://twitter.com/Unpop_Science/status/15816...,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,1e19d973-1b72-0c02-8f0c-aa9fa68946d8,30c79b39-7730-008f-8d2a-6ead5a286cb9,ebcc9b12-9efe-0e6a-927f-02cc5ffc3d65,spencer 🦈,Gaia4,311,WHAT KILLED THE CRABS?\n\nThe official story i...,twitter,14,Sunday,2022-10-16
1652,f9735a7d-0207-03b6-8ce3-dd800aed9ec6,messages,2022-10-16 14:56:08.000,"As sea ice forms in winter, salt is expelled a...",https://twitter.com/Unpop_Science/status/15816...,twitter,b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00...,1e19d973-1b72-0c02-8f0c-aa9fa68946d8,30

In [44]:
from anthropic import Anthropic
import os

def call_haiku_api(message: str):
    client = Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
    
    response = client.messages.create(
        model='claude-3-5-haiku-latest',
        max_tokens=1024,
        messages=[{
            'role': 'user',
            'content': message
        }]
    )
    
    return response.content[0].text

In [45]:
call_haiku_api("Hello")

'Hi there! How are you doing today? Is there anything I can help you with?'

In [46]:
df_memories.to_csv('memories.csv', index=False)

In [50]:
df_memories['agent_name'].unique()

array(['Gaia4', 'Astraea', 'Genesis', 'Cascadia', 'Nexus', 'Aquarius',
       'TerraNova'], dtype=object)

In [67]:
import panel as pn
import param
import pandas as pd
from datetime import datetime
pn.extension()

class RoomExplorer(param.Parameterized):
    agent = param.Selector(objects=list(df_memories['agent_name'].unique()))
    source = param.Selector(objects=sources)
    room_id = param.Selector()
   
    def __init__(self, df_memories, **params):
        self.df_memories = df_memories
        super().__init__(**params)

    @param.depends('agent', 'source', on_init=True, watch=True)
    def update_room_ids(self):
        filtered_df = self.df_memories[
            (self.df_memories['agent_name'] == self.agent) & 
            (self.df_memories['source'] == self.source)
        ]
        self.room_ids = filtered_df['roomId'].unique()
        self.param.room_id.objects = self.room_ids
        try:
            self.room_id = self.room_ids[0]
        except:
            pass
       
    def get_room_stats(self):
        room_data = self.df_memories[self.df_memories['roomId'] == self.room_id]
        
        stats = {
           'Source': room_data['source'].iloc[0],
           'Total Memories': len(room_data),
           'Unique Participants': room_data['user_name'].nunique(),
           'Unique Agents': room_data['agent_name'].nunique(),
           'Date Range': f"{room_data['createdAt'].min()} to {room_data['createdAt'].max()}"
        }
        
        return pn.pane.HTML(f"""
        <div style="padding: 10px; background: #f5f5f5; border-radius: 5px;">
           <h3>Room Statistics</h3>
           {''.join(f'<p><b>{k}:</b> {v}</p>' for k,v in stats.items())}
        </div>
        """)
   
    def get_messages(self):
        room_data = self.df_memories[self.df_memories['roomId'] == self.room_id]
        room_data = room_data.sort_values('createdAt')
        
        def row_link_name(row):
            if row['source'] == 'twitter':
                return row['url'].split('/')[3]
            elif row['source'] == 'discord':
                return row['user_name'] or row['agent_name']
            elif row['source'] == 'direct':
                return 'direct'
            else:
                return row['user_name'] or row['agent_name']
        
        messages_html = ''.join([
           f"""
           <div style="margin: 10px 0; padding: 10px; border: 1px solid #eee;">
               <p><b><a href={row['url']}>{row_link_name(row)}</a></b> at {row['createdAt']} </p>
               <p>{row['text']}</p>
           </div>
           """ for _, row in room_data.iterrows()
        ])
        
        return pn.pane.HTML(f"<div style='height: 500px; overflow-y: scroll;'>{messages_html}</div>")
   
    @param.depends('room_id')
    def view(self):
       if not self.room_id:
           return pn.pane.HTML("Select a room to begin")
           
       return pn.Column(
           self.get_room_stats(),
           self.get_messages()
       )

# Initialize and display
explorer = RoomExplorer(df_memories)
pn.Row(explorer.param, explorer.view).servable()

Row
    [0] Column(margin=(5, 10), name='RoomExplorer')
        [0] StaticText(value='<b>RoomExplorer</b>')
        [1] Select(name='Agent', options=OrderedDict([('Gaia4', ...]), value='Gaia4')
        [2] Select(name='Source', options=OrderedDict([('echochamber...]), value='echochambers')
        [3] Select(name='Room id', options=OrderedDict([('dfe2db74-97...]), value='dfe2db74-975e-0aa9-b6fd-d...)
    [1] ParamMethod(method, _pane=Column, defer_load=False)